# Crew for Research Analysts

This notebook provides a comprehensive comparison of different CrewAI configurations for tackling research and summarization tasks. The primary goal is to demonstrate how varying levels of agent sophistication, tool integration, and multi-agent collaboration impact the quality and depth of generated outputs.

We will explore three distinct setups, all focused on summarizing 'Changes in life expectancy in India from 1600s to 2026':

1.  **Simple Crew Setup (No Tools):** A basic single-agent configuration that summarizes a topic based solely on its internal knowledge, without access to external information.

2.  **Advanced Web Search Crew Setup (With Tools):** A single-agent setup equipped with web search and scraping tools (`SerperDevTool`,
Research Analyst with CrewAI.ipynb
Research Analyst with CrewAI.ipynb_

In [ ]:
!pip install -q crewai litellm
!apt-get update
!apt-get install -y zstd

!curl -fsSL https://ollama.com/install.sh | sh
!pip install crewai[SerperDevTool] crewai[ScrapeWebsiteTool] crewai_tools

In [ ]:
import subprocess
subprocess.Popen(["ollama", "serve"])

<Popen: returncode: None args: ['ollama', 'serve']>

In [ ]:
!ollama pull gemma4:e4b

In [ ]:
import os
from crewai import Agent, Task, Crew, LLM, Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

llm = LLM(model="ollama/gemma4:e4b", base_url="http://localhost:11434")

### 1. Simpler Crew Setup: One Agent, One Task, No Tools

Let's create a simpler setup with just one agent and one task, without any tools, to see how the output differs. This agent will act as a basic summarizer.

In [ ]:
from crewai import Agent, Task, Crew, Process

simple_summarizer_agent = Agent(
    role='Basic Summarizer',
    goal='Provide a concise summary of the given topic without external research.',
    backstory="""You are a simple summarizer, capable of taking a prompt and providing a direct, brief summary based solely on your internal knowledge.
    You do not have access to external tools or data sources.""",
    verbose=True,
    allow_delegation=False,
    llm=llm # Using the same LLM as before
)

simple_summary_task = Task(
    description=(
        "Summarize the following topic: {topic}. "
        "Provide a brief overview, highlighting key aspects and current trends."
    ),
    expected_output="A concise, high-level summary of the provided topic.",
    agent=simple_summarizer_agent
)

simple_crew = Crew(
    agents=[simple_summarizer_agent],
    tasks=[simple_summary_task],
    process=Process.sequential,
    verbose=True
)

Now, let's run this simpler crew with the same topic and compare the results.

In [ ]:
simple_results = simple_crew.kickoff(inputs={
    'topic': 'Changes in life expectancy in India from 1600s to 2026'
})

print("\n--- Simple Crew Results ---\n")
print(simple_results)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 54e7841d-0fd0-47cc-a6b7-aedb7481beec                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Summarize the following topic: Changes in life expectancy in India from 1600s to 2026. Provide a brief   │
│  overview, highlighting key aspects and current trends.                                                         │
│  ID: 52b11ed6-7d28-4f3c-97f3-c6602ab92862                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Basic Summarizer                                                                                        │
│                                                                                                                 │
│  Task: Summarize the following topic: Changes in life expectancy in India from 1600s to 2026. Provide a brief   │
│  overview, highlighting key aspects and current trends.                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Basic Summarizer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Life expectancy in India has shown a trajectory of significant but uneven growth from the 1600s to projected   │
│  levels in 2026. In the 1600s, life expectancy was severely constrained by high rates of endemic disease,       │
│  famine, and poor sanitation, keeping overall life spans low and highly volatile year over year.                │
│                                                                                                                 │
│  The most pronounced gains began in the 20th century, driven initially by improved public health                │
│  infrastructure, vaccination programs, and better nutritional practices, leading to a reduction in infant and   │
│  child mortality. From the late 20th century onwards, the growth rate has continued, albeit meeting new         │
│  challenges.                                                                                                    │
│                                                                                                                 │
│  Current trends indicate a steady, yet slowing, improvement, characterized by better access to healthcare and   │
│  increased educational attainment. However, this progress is tempered by modern issues such as                  │
│  non-communicable diseases (NCDs) like heart disease and diabetes, environmental pollution, and persistent      │
│  socio-economic disparities between regions and castes. By 2026, projections suggest continued moderate gains,  │
│  contingent heavily upon sustained investment in robust public health systems and addressing underlying         │
│  determinants of health equity.                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Summarize the following topic: Changes in life expectancy in India from 1600s to 2026. Provide a brief   │
│  overview, highlighting key aspects and current trends.                                                         │
│  Agent: Basic Summarizer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 54e7841d-0fd0-47cc-a6b7-aedb7481beec                                                                       │
│  Final Output: Life expectancy in India has shown a trajectory of significant but uneven growth from the 1600s  │
│  to projected levels in 2026. In the 1600s, life expectancy was severely constrained by high rates of endemic   │
│  disease, famine, and poor sanitation, keeping overall life spans low and highly volatile year over year.       │
│                                                                                                                 │
│  The most pronounced gains began in the 20th century, driven initially by improved public health                │
│  infrastructure, vaccination programs, and better nutritional practices, leading to a reduction in infant and   │
│  child mortality. From the late 20th century onwards, the growth rate has continued, albeit meeting new         │
│  challenges.                                                                                                    │
│                                                                                                                 │
│  Current trends indicate a steady, yet slowing, improvement, characterized by better access to healthcare and   │
│  increased educational attainment. However, this progress is tempered by modern issues such as                  │
│  non-communicable diseases (NCDs) like heart disease and diabetes, environmental pollution, and persistent      │
│  socio-economic disparities between regions and castes. By 2026, projections suggest continued moderate gains,  │
│  contingent heavily upon sustained investment in robust public health systems and addressing underlying         │
│  determinants of health equity.                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- Simple Crew Results ---

Life expectancy in India has shown a trajectory of significant but uneven growth from the 1600s to projected levels in 2026. In the 1600s, life expectancy was severely constrained by high rates of endemic disease, famine, and poor sanitation, keeping overall life spans low and highly volatile year over year.

The most pronounced gains began in the 20th century, driven initially by improved public health infrastructure, vaccination programs, and better nutritional practices, leading to a reduction in infant and child mortality. From the late 20th century onwards, the growth rate has continued, albeit meeting new challenges.

Current trends indicate a steady, yet slowing, improvement, characterized by better access to healthcare and increased educational attainment. However, this progress is tempered by modern issues such as non-communicable diseases (NCDs) like heart disease and diabetes, environmental pollution, and persistent socio-economic disparities be

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 2. Advanced Crew Setup: One Agent, One Task, with Web Search Tools

Now, let's create an advanced summarizer that uses web search tools to gather information before summarizing. This will bridge the gap between the simple summarizer and the multi-agent detailed analysis.

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

web_search_summarizer_agent = Agent(
    role='Advanced Web Search Summarizer',
    goal='Provide a comprehensive summary of the given topic using external web search and scraping.',
    backstory="""You are an expert summarizer equipped with powerful web search and scraping tools. You can find and synthesize information from the internet to provide a detailed and well-researched summary of any given topic.""",
    tools=[search_tool, scrape_tool],
    verbose=True,
    allow_delegation=False,
    llm=llm
)

web_search_summary_task = Task(
    description=(
        "Research and summarize the following topic: {topic}. "
        "Use your web search tools to gather relevant information, then synthesize it into a comprehensive summary, highlighting key aspects, historical context, and current trends."
        "Ensure the summary is well-structured and cites sources if appropriate."
    ),
    expected_output="A comprehensive, well-researched summary of the provided topic, incorporating information from web sources.",
    agent=web_search_summarizer_agent
)

web_search_crew = Crew(
    agents=[web_search_summarizer_agent],
    tasks=[web_search_summary_task],
    process=Process.sequential,
    verbose=True
)

Let's run this web-search-enabled crew and compare its results with both the simple summarizer and the detailed crew.

In [ ]:
web_search_results = web_search_crew.kickoff(inputs={
    'topic': 'Changes in life expectancy in India from 1600s to 2026'
})

print("\n--- Advanced Web Search Crew Results ---\n")
print(web_search_results)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3399977d-849c-4b1d-b75d-368befd2643d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research and summarize the following topic: Changes in life expectancy in India from 1600s to 2026. Use  │
│  your web search tools to gather relevant information, then synthesize it into a comprehensive summary,         │
│  highlighting key aspects, historical context, and current trends.Ensure the summary is well-structured and     │
│  cites sources if appropriate.                                                                                  │
│  ID: 88c0f024-0225-48ec-908e-90241258f36f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Advanced Web Search Summarizer                                                                          │
│                                                                                                                 │
│  Task: Research and summarize the following topic: Changes in life expectancy in India from 1600s to 2026. Use  │
│  your web search tools to gather relevant information, then synthesize it into a comprehensive summary,         │
│  highlighting key aspects, historical context, and current trends.Ensure the summary is well-structured and     │
│  cites sources if appropriate.                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'life expectancy changes in India from 1600s to 2026'}                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': 'life expectancy changes in India from 1600s to 2026', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'India: life expectancy 1800-2020 - Statista',...


╭─────────────────────────────────────── ✅ Tool Execution Completed (#3) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Output: {'searchParameters': {'q': 'life expectancy changes in India from 1600s to 2026', 'type': 'search',    │
│  'num': 10, 'engine': 'google'}, 'organic': [{'title': 'India: life expectancy 1800-2020 - Statista', 'link':   │
│  'https://www.statista.com/statistics/1041383/life-expectancy-india-all-time/?srsltid=AfmBOorFJdvvQFpiDEN_dikZ  │
│  OBCceuqR8XPL4IlPnv_FjhzlfWGkJrQv', 'snippet': 'Life expectancy in India was 25.4 in the year 1800, and over    │
│  the course of the next 220 years, it has increased to almost 70.', 'position': 1}, {'title': 'India Life       │
│  Expectancy (1950-2025) - Macrotrends', 'link':                                                                 │
│  'https://www.macrotrends.net/global-metrics/countries/ind/india/life-expectancy', 'snippet': 'India life       │
│  expectancy for 2025 was 70.82, a 0.29% increase from 2024. India life expectancy for 2024 was 70.62, a 1.92%   │
│  decline from 2023.', 'position': 2}, {'title': "India's life expectancy has gone up from 48 in 1965 to 73 now  │
│  - Reddit", 'link':                                                                                             │
│  'https://www.reddit.com/r/IndiaStatistics/comments/1qusfg3/indias_life_expectancy_has_gone_up_from_48_in/',    │
│  'snippet': "India's life expectancy has gone up from 48 in 1965 to 73 now.", 'position': 3}, {'title': 'Life   │
│  expectancy - Our World in Data', 'link': 'https://ourworldindata.org/grapher/life-expectancy', 'snippet':      │
│  'Across the world, people are living longer. In 1900, the global average life expectancy was 32 years. By      │
│  2023, this had more than doubled to ...', 'position': 4}, {'title': 'What is the history of life expectancy    │
│  in India before and ... - Quora', 'link':                                                                      │
│  'https://www.quora.com/What-is-the-history-of-life-expectancy-in-India-before-and-after-the-advent-of-the-Mug  │
│  hals-Alternatively-what-are-some-data-sources-for-knowing-the-average-Indian-lifespan-during-Medieval-times',  │
│   'snippet': 'In India and China, life expectancy has risen by 30 years since 1950. Even in Africa, life        │
│  expectancy rose by 13 years from the early 1950s ...', 'position': 5, 'sitelinks': [{'title': 'What was the    │
│  life expectancy of people in ancient India? - Quora', 'link':                                                  │
│  'https://www.quora.com/What-was-the-life-expectancy-of-people-in-ancient-India'}, {'title': 'What is the life  │
│  expectancy of a person in India from the 18th century ...', 'link':                                            │
│  'https://www.quora.com/What-is-the-life-expectancy-of-a-person-in-India-from-the-18th-century-to-date'}]},     │
│  {'title': 'Demographics of India - Wikipedia', 'link': 'https://en.wikipedia.org/wiki/Demographics_of_India',  │
│  'snippet': '70.52 years (2023 est.) · 73.60 years (2023 est.) · Decrease 1.90 children born per woman (2025)   │
│  · Positive decrease 22.6 deaths/1,000 live births (2023).', 'position': 6}, {'title': 'Life expectancy was     │
│  69.7… | Anagh Pal | 10 comments - LinkedIn', 'link':                                                           │
│  'https://www.linkedin.com/posts/anagh_indias-life-expectancy-over-the-years-activity-7252871482039554049-amH5  │
│  ', 'snippet': "India's life expectancy over the years. 1800: Life expectancy was 25.4 years. 1950: Life        │
│  expectancy was 35.21 years. 2000: Life expectancy ..."

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Advanced Web Search Summarizer                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Evolution of Life Expectancy in India: A Historical Overview (1600s to Projections for 2026)                │
│                                                                                                                 │
│  Life expectancy in India has undergone a dramatic and complex transformation over the last few centuries.      │
│  While concrete, consistent data from the 17th century is scarce due to record-keeping limitations, available   │
│  trends—spanning colonial records, post-independence statistics, and modern projections—illustrate a            │
│  significant trajectory of improvement in health and longevity.                                                 │
│                                                                                                                 │
│  ### I. The Historical Context (1600s to Early 20th Century)                                                    │
│                                                                                                                 │
│  Data from the early modern period often reflects the challenges inherent in resource management, epidemic      │
│  outbreaks, and varying levels of sanitation.                                                                   │
│                                                                                                                 │
│  *   **18th and Early 19th Century Estimates:** Records available from periods like the 1800s suggest a marked  │
│  increase from baseline estimates. One source indicates that life expectancy in **1800 was approximately 25.4   │
│  years** (Statista). Furthermore, colonial-era data hints at fluctuations, such as a documented dip in British  │
│  India’s life expectancy from an estimated 25 years in 1885 to 21 years in 1920 (Instagram snippet).            │
│  *   **Early 20th Century Trends:** Life expectancy across the subcontinent was highly susceptible to major     │
│  health crises. While overall improvement was occurring, these periods were marked by significant               │
│  vulnerability to public health challenges.                                                                     │
│                                                                                                                 │
│  ### II. The Mid-20th Century Transition (1940s – 1970s)                                                        │
│                                                                                                                 │
│  The post-independence period marked a crucial turning point, initiating sustained improvements despite         │
│  economic hurdles.                                                                                              │
│                                                                                                                 │
│  *   **The Post-War Years:** By the 1950s, estimates suggest a gradual upward climb. Some records place life    │
│  expectancy around **35 years** in the early 1950s (Facebook snippet).                                          │
│  *   **Mid-Century Growth:** A specific data point suggests a notable increase from an estimated **48 years in  │
│  1965** (Reddit snippet). By the 1970s, indicators show further upward movement, with one observation noting    │
│  life expectancy around **49.7 years in 1970–75** (Econ

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Research and summarize the following topic: Changes in life expectancy in India from 1600s to 2026. Use  │
│  your web search tools to gather relevant information, then synthesize it into a comprehensive summary,         │
│  highlighting key aspects, historical context, and current trends.Ensure the summary is well-structured and     │
│  cites sources if appropriate.                                                                                  │
│  Agent: Advanced Web Search Summarizer                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- Advanced Web Search Crew Results ---

## Evolution of Life Expectancy in India: A Historical Overview (1600s to Projections for 2026)

Life expectancy in India has undergone a dramatic and complex transformation over the last few centuries. While concrete, consistent data from the 17th century is scarce due to record-keeping limitations, available trends—spanning colonial records, post-independence statistics, and modern projections—illustrate a significant trajectory of improvement in health and longevity.

### I. The Historical Context (1600s to Early 20th Century)

Data from the early modern period often reflects the challenges inherent in resource management, epidemic outbreaks, and varying levels of sanitation.

*   **18th and Early 19th Century Estimates:** Records available from periods like the 1800s suggest a marked increase from baseline estimates. One source indicates that life expectancy in **1800 was approximately 25.4 years** (Statista). Furthermore, colonial-era dat

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3399977d-849c-4b1d-b75d-368befd2643d                                                                       │
│  Final Output: ## Evolution of Life Expectancy in India: A Historical Overview (1600s to Projections for 2026)  │
│                                                                                                                 │
│  Life expectancy in India has undergone a dramatic and complex transformation over the last few centuries.      │
│  While concrete, consistent data from the 17th century is scarce due to record-keeping limitations, available   │
│  trends—spanning colonial records, post-independence statistics, and modern projections—illustrate a            │
│  significant trajectory of improvement in health and longevity.                                                 │
│                                                                                                                 │
│  ### I. The Historical Context (1600s to Early 20th Century)                                                    │
│                                                                                                                 │
│  Data from the early modern period often reflects the challenges inherent in resource management, epidemic      │
│  outbreaks, and varying levels of sanitation.                                                                   │
│                                                                                                                 │
│  *   **18th and Early 19th Century Estimates:** Records available from periods like the 1800s suggest a marked  │
│  increase from baseline estimates. One source indicates that life expectancy in **1800 was approximately 25.4   │
│  years** (Statista). Furthermore, colonial-era data hints at fluctuations, such as a documented dip in British  │
│  India’s life expectancy from an estimated 25 years in 1885 to 21 years in 1920 (Instagram snippet).            │
│  *   **Early 20th Century Trends:** Life expectancy across the subcontinent was highly susceptible to major     │
│  health crises. While overall improvement was occurring, these periods were marked by significant               │
│  vulnerability to public health challenges.                                                                     │
│                                                                                                                 │
│  ### II. The Mid-20th Century Transition (1940s – 1970s)                                                        │
│                                                                                                                 │
│  The post-independence period marked a crucial turning point, initiating sustained improvements despite         │
│  economic hurdles.                                                                                              │
│                                                                                                                 │
│  *   **The Post-War Years:** By the 1950s, estimates suggest a gradual upward climb. Some records place life    │
│  expectancy around **35 years** in the early 1950s (Facebook snippet).                                          │
│  *   **Mid-Century Growth:** A specific data point suggests a notable increase from an estimated **48 years in  │
│  1965** (Reddit snippet). By the 1970s, indicators show further upward movement, with one observation noting    │
│  life expectancy around **49.7 years in 1970–75** (Eco

### 3. Multi-Agent System

This setup represented our most sophisticated approach, employing a multi-agent system to tackle the research and analysis of life expectancy in India. It consisted of two specialized agents:

1.  **Lead Data Acquisition Specialist (`data_collector`):** This agent was equipped with `SerperDevTool` and `ScrapeWebsiteTool` to perform extensive web searches, identify relevant data, and verify sources. Its goal was to gather a comprehensive raw dataset.

2.  **Principal Insights Architect (`data_analyst`):** This agent's role was to receive the raw data from the `data_collector`, perform trend analysis, identify primary drivers, and synthesize all findings into a final, detailed analytical report. This agent had the ability to delegate back to the data collector if more information was needed.

**Process:** The agents worked sequentially (`Process.sequential`), ensuring that data collection was completed before analysis began. The `analysis_task` had `collection_task` as its context, allowing the analyst to build upon the collector's work.

**Outcome:** This multi-agent system produced the most comprehensive and structured output, saved as `universal_analysis_report.md`. This report included an executive summary, detailed trend breakdowns, and a definitive conclusion, demonstrating the power of specialized agents collaborating on complex research and analytical tasks.

In [ ]:
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

data_collector = Agent(
    role='Lead Data Acquisition Specialist',
    goal='Identify, fetch, and verify the most relevant information regarding {topic} in the {domain} sector.',
    backstory="""You are a master of digital intelligence. Your objective is to
    navigate the internet using advanced search queries to find high-authority
    sources, raw datasets, and expert opinions. You provide the foundation
    of truth for any given subject without bias.""",
    tools=[search_tool, scrape_tool],
    verbose=True,
    allow_delegation=False, # Focuses strictly on gathering data
    llm=llm,
    # memory=True
)

data_analyst = Agent(
    role='Principal Insights Architect',
    goal='Transform raw data into a structured, high-level summary and strategic conclusion.',
    backstory="""You are a world-class analyst capable of seeing patterns in
    disparate data points. Whether the data is financial, historical, or
    technical, you specialize in extracting the "so what?"—explaining
    trends, identifying anomalies, and providing a definitive conclusion
    based on the evidence provided.""",
    verbose=True,
    allow_delegation=True, # Can ask the collector to find missing data points
    llm=llm,
    # memory=True
)



collection_task = Task(
    description=(
        "Conduct a deep dive into {topic} within the {domain} industry. "
        "1. Search for key figures, metrics, and qualitative insights from the last decade to 2026. "
        "2. Cross-reference multiple sources to ensure data integrity. "
        "3. Organize the findings into a clear, raw data report for analysis."
    ),
    expected_output="A comprehensive raw dataset containing facts, figures, and verified sources.",
    agent=data_collector
)

analysis_task = Task(
    description=(
        "Review the raw data provided regarding {topic}. "
        "1. Perform a trend analysis to show how the subject has evolved. "
        "2. Identify the primary drivers behind the observed data. "
        "3. Synthesize everything into a final, detailed summary that provides "
        "a clear conclusion on the current and future state of {topic}."
    ),
    expected_output="A professional analytical report with an executive summary, trend breakdown, and conclusion.",
    agent=data_analyst,
    context=[collection_task],
    output_file="universal_analysis_report.md"
)


universal_crew = Crew(
    agents=[data_collector, data_analyst],
    tasks=[collection_task, analysis_task],
    process=Process.sequential,
    verbose=True
)


results = universal_crew.kickoff(inputs={
    'topic': 'Changes in life expectancy in India from 1600s to 2026',
    'domain': 'Public Health and Demographics'
})

print(results)

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e2e48f30-2f01-4c24-92d3-83220850d981                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Conduct a deep dive into Changes in life expectancy in India from 1600s to 2026 within the Public        │
│  Health and Demographics industry. 1. Search for key figures, metrics, and qualitative insights from the last   │
│  decade to 2026. 2. Cross-reference multiple sources to ensure data integrity. 3. Organize the findings into a  │
│  clear, raw data report for analysis.                                                                           │
│  ID: 2299d322-42d7-46e8-9cb0-680823f449de                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Data Acquisition Specialist                                                                        │
│                                                                                                                 │
│  Task: Conduct a deep dive into Changes in life expectancy in India from 1600s to 2026 within the Public        │
│  Health and Demographics industry. 1. Search for key figures, metrics, and qualitative insights from the last   │
│  decade to 2026. 2. Cross-reference multiple sources to ensure data integrity. 3. Organize the findings into a  │
│  clear, raw data report for analysis.                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'historical deep dive life expectancy India 1600 to 2026 public health datasets WHO     │
│  World Bank academia'}                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Review the raw data provided regarding Changes in life expectancy in India from 1600s to 2026. 1.        │
│  Perform a trend analysis to show how the subject has evolved. 2. Identify the primary drivers behind the       │
│  observed data. 3. Synthesize everything into a final, detailed summary that provides a clear conclusion on     │
│  the current and future state of Changes in life expectancy in India from 1600s to 2026.                        │
│  ID: e05494e0-9e31-4c50-9cb6-8597ccb2a5e5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Principal Insights Architect                                                                            │
│                                                                                                                 │
│  Task: Review the raw data provided regarding Changes in life expectancy in India from 1600s to 2026. 1.        │
│  Perform a trend analysis to show how the subject has evolved. 2. Identify the primary drivers behind the       │
│  observed data. 3. Synthesize everything into a final, detailed summary that provides a clear conclusion on     │
│  the current and future state of Changes in life expectancy in India from 1600s to 2026.                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Principal Insights Architect                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **ANALYSIS REPORT: EVOLUTION OF LIFE EXPECTANCY IN INDIA (1600s – 2026)**                                      │
│                                                                                                                 │
│  ***                                                                                                            │
│                                                                                                                 │
│  ### **EXECUTIVE SUMMARY**                                                                                      │
│                                                                                                                 │
│  This report analyzes the projected and historical trajectory of life expectancy in India spanning from the     │
│  17th century through 2026. The evidence reveals one of the most dramatic and positive demographic shifts in    │
│  recorded history. India has transitioned from a low-survivability, pre-industrial life expectancy (estimated   │
│  in the lower 25 years in the early 1800s) to a contemporary status approximating 60–70 years. This growth      │
│  trajectory is not linear but exponential, marked by discontinuous leaps correlated with institutional,         │
│  medical, and socioeconomic advancements.                                                                       │
│                                                                                                                 │
│  The primary drivers powering this increase are the modernization of public health infrastructure,              │
│  improvements in sanitation, nutritional security, and changes in medical practice. While the trend shown is    │
│  unequivocally positive, the analysis concludes that the projections for 2026 must be viewed with critical      │
│  awareness. Future gains, while anticipated, will depend heavily on sustaining robust investments in primary    │
│  healthcare, combating non-communicable diseases, and achieving comprehensive developmental parity across all   │
│  demographic strata.                                                                                            │
│                                                                                                                 │
│  ***                                                                                                            │
│                                                                                                                 │
│  ### **I. TREND BREAKDOWN: THE EVOLUTIONARY ARC (1600s – 2026)**                                                │
│                                                                                                                 │
│  The data progression can be segmented into distinct developmental phases, revealing a clear upward curve       │
│  against a backdrop of substantial data gaps.                                                                   │
│                                                                                                                 │
│  **A. Pre-Modern & Early Modern Period (1600s – Mid-1800s): The Baseline Era**                                  │
│  * **Data Profile:** Data continuity is exceptionally weak. Life expectancy during the 1600s is more            │
│  accurately represented by scholarly reconstruction rat

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the raw data provided regarding Changes in life expectancy in India from 1600s to 2026. 1.        │
│  Perform a trend analysis to show how the subject has evolved. 2. Identify the primary drivers behind the       │
│  observed data. 3. Synthesize everything into a final, detailed summary that provides a clear conclusion on     │
│  the current and future state of Changes in life expectancy in India from 1600s to 2026.                        │
│  Agent: Principal Insights Architect                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**ANALYSIS REPORT: EVOLUTION OF LIFE EXPECTANCY IN INDIA (1600s – 2026)**

***

### **EXECUTIVE SUMMARY**

This report analyzes the projected and historical trajectory of life expectancy in India spanning from the 17th century through 2026. The evidence reveals one of the most dramatic and positive demographic shifts in recorded history. India has transitioned from a low-survivability, pre-industrial life expectancy (estimated in the lower 25 years in the early 1800s) to a contemporary status approximating 60–70 years. This growth trajectory is not linear but exponential, marked by discontinuous leaps correlated with institutional, medical, and socioeconomic advancements.

The primary drivers powering this increase are the modernization of public health infrastructure, improvements in sanitation, nutritional security, and changes in medical practice. While the trend shown is unequivocally positive, the analysis concludes that the projections for 2026 must be viewed with critical awaren

### Saving and Displaying All Crew Results for Comparison

In [ ]:
# Save Simple Crew Results to a file
with open('simple_crew_report.md', 'w') as f:
    f.write(simple_results.raw)

# Save Advanced Web Search Crew Results to a file
with open('web_search_crew_report.md', 'w') as f:
    f.write(web_search_results.raw)